# World Cup Shorts — Dataset Exploration

Visualizes all source datasets and runs `correlation_to_winning()`.
Run each section after downloading CSVs to `data/`.

**Setup:** `pip install pandas numpy matplotlib seaborn scipy`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
from pipeline import (
    load_international, load_matches, load_group_standings,
    load_manager_appearances, load_elo,
    knockout_stage_score, group_stage_points, elo_at_wc_start, coach_wc_experience,
    build_feature_matrix, correlation_to_winning,
    WC_2026_TEAMS
)

plt.rcParams.update({
    'figure.facecolor': '#1e1e1e',
    'axes.facecolor':   '#1e1e1e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'text.color':       '#ccc',
    'xtick.color':      '#ccc',
    'ytick.color':      '#ccc',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
})
ACCENT = '#00ff88'
RED    = '#D32531'

print('Setup OK')

---
## Dataset 1 — International Results (martj42, 1872–2026)
49k+ matches. Source of head-to-head records and win rates.

In [ ]:
intl = load_international()
print(f'Rows: {len(intl):,}  |  Columns: {intl.columns.tolist()}')
intl.head(3)

In [ ]:
# WC-only subset
wc_intl = intl[intl['tournament'] == 'FIFA World Cup'].copy()
print(f'WC matches: {len(wc_intl):,} | Editions: {wc_intl["date"].dt.year.nunique()}')

# Win rate per team in WC matches
def wc_win_rate(df, team):
    home = df[df['home_team'] == team].copy()
    away = df[df['away_team'] == team].copy()
    home['win'] = home['home_score'] > home['away_score']
    away['win'] = away['away_score'] > away['home_score']
    total = pd.concat([home[['win']], away[['win']]])
    return total['win'].mean() if len(total) > 0 else None

win_rates = {t: wc_win_rate(wc_intl, t) for t in WC_2026_TEAMS}
wr_df = pd.DataFrame({'team': list(win_rates.keys()), 'win_rate': list(win_rates.values())})
wr_df = wr_df.dropna().sort_values('win_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [ACCENT if wr_df.iloc[i]['win_rate'] >= 0.5 else '#555' for i in range(len(wr_df))]
bars = ax.barh(wr_df['team'], wr_df['win_rate'], color=colors, height=0.7)
ax.axvline(0.5, color=RED, linewidth=1.2, linestyle='--', label='50% threshold')
ax.set_xlabel('Win Rate in WC Matches (all editions)')
ax.set_title('WC Win Rate — 2026 Teams', color=ACCENT, fontsize=14, pad=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.savefig('data/viz_wc_win_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Goals scored per game trend by decade
wc_intl['decade'] = (wc_intl['date'].dt.year // 10 * 10).astype(str) + 's'
wc_intl['total_goals'] = wc_intl['home_score'] + wc_intl['away_score']
goals_by_decade = wc_intl.groupby('decade')['total_goals'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(goals_by_decade.index, goals_by_decade.values, color=ACCENT, linewidth=2.5, marker='o')
ax.fill_between(goals_by_decade.index, goals_by_decade.values, alpha=0.15, color=ACCENT)
ax.set_title('Avg Goals per WC Match by Decade', color=ACCENT, fontsize=13, pad=10)
ax.set_ylabel('Goals per match')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('data/viz_goals_by_decade.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Dataset 2 — jfjelstul World Cup Database (1930–2018)
27 tables. Stage reached, group standings, manager experience.

In [ ]:
matches = load_matches()
print(f'matches: {len(matches):,} rows')
print(f'stage_name values (knockout): {matches[matches["knockout_stage"]==1]["stage_name"].unique()}')

ko_df     = knockout_stage_score(matches)
group_pts = group_stage_points(matches)

print(f'\nko_df sample:\n{ko_df.head(6)}')
print(f'\ngroup_pts sample:\n{group_pts.head(6)}')



In [ ]:
# Heatmap: team × WC year → knockout score
pivot = group_pts[group_pts['team'].isin(WC_2026_TEAMS)].pivot_table(
    index='team', columns='year', values='group_points', aggfunc='max'
)

fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(
    pivot, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='#2a2a2a',
    vmin=0, vmax=6,
    cbar_kws={'label': 'Group score (0=DNQ … 6=Champion)'},
    annot=True, fmt='.0f', annot_kws={'size': 7, 'color': '#eee'}
)
ax.set_title('Group Score per WC Edition — 2026 Teams', color=ACCENT, fontsize=14, pad=12)
ax.set_xlabel('WC Year')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('data/viz_ko_score_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
matches  = load_matches()

ko_df     = knockout_stage_score(matches)
group_pts = group_stage_points(matches)


print(f'matches: {len(matches):,} rows | columns: {matches.columns.tolist()}')
print(f'\nko_df sample:\n{ko_df.head(6)}')
print(f'\ngroup_pts sample:\n{group_pts.head(6)}')

# Heatmap: team × WC year → knockout score
pivot = ko_df[ko_df['team'].isin(WC_2026_TEAMS)].pivot_table(
    index='team', columns='year', values='ko_score', aggfunc='max'
)

score_labels = {0: 'DNQ', 1: 'R32', 2: 'R16', 3: 'QF', 4: 'SF', 5: 'Final', 6: 'Won'}

fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(
    pivot, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='#2a2a2a',
    vmin=0, vmax=6,
    cbar_kws={'label': 'KO score (0=DNQ … 6=Champion)'},
    annot=True, fmt='.0f', annot_kws={'size': 7, 'color': '#eee'}
)
ax.set_title('Knockout Score per WC Edition — 2026 Teams', color=ACCENT, fontsize=14, pad=12)
ax.set_xlabel('WC Year')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('data/viz_ko_score_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Assuming you already ran: ko_df = knockout_stage_score(matches)

# 1. Filter for only the rows where a team achieved a score of 6
champions_df = ko_df[ko_df["ko_score"] == 6]

# 2. Count the occurrences of each team
title_counts = champions_df["team"].value_counts().reset_index()
title_counts.columns = ["Team", "Times Score 6 (Titles)"]

# 3. Display the results
print(title_counts.to_string(index=False))

In [ ]:
# Champions vs finalists
champions  = ko_df[ko_df['ko_score'] == 6].groupby('team').size().reset_index(name='titles')
finalists  = ko_df[ko_df['ko_score'] >= 5].groupby('team').size().reset_index(name='finals')
combined   = finalists.merge(champions, on='team', how='left').fillna(0)
combined   = combined[combined['team'].isin(WC_2026_TEAMS)].sort_values('finals')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(combined['team'], combined['finals'],  color='#555',  height=0.6, label='Finals reached')
ax.barh(combined['team'], combined['titles'],  color=ACCENT,  height=0.6, label='Titles won')
ax.set_title('WC Finals & Titles (1930–2018) — 2026 Teams', color=ACCENT, fontsize=13, pad=10)
ax.set_xlabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('data/viz_finals_titles.png', dpi=150, bbox_inches='tight')
plt.show()

# Group stage points distribution per team
gp_2026 = group_pts[group_pts['team'].isin(WC_2026_TEAMS)]
gp_avg  = gp_2026.groupby('team')['group_points'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(gp_avg.index, gp_avg.values, color=ACCENT, height=0.7)
ax.set_title('Avg Group Stage Points per WC (1930–2018) — 2026 Teams',
             color=ACCENT, fontsize=13, pad=10)
ax.set_xlabel('Avg group stage points')
plt.tight_layout()
plt.savefig('data/viz_group_points.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Manager experience distribution
try:
    mgr = load_manager_appearances()
    print(f'Manager appearances: {len(mgr):,} | Columns: {mgr.columns.tolist()}')
    exp = coach_wc_experience(mgr)

    fig, ax = plt.subplots(figsize=(8, 4))
    exp['prior_wc_count'].value_counts().sort_index().plot(
        kind='bar', ax=ax, color=ACCENT, edgecolor='#1e1e1e'
    )
    ax.set_title('Distribution of Coach Prior WC Experience', color=ACCENT, fontsize=13, pad=10)
    ax.set_xlabel('Prior WC tournaments managed')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('data/viz_coach_experience.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Skipped (check column names): {e}')

---
## Dataset 3 — ELO Ratings (Kaggle, afonsofernandescruz)
Historical ELO per team per date. Most predictive single feature.

In [ ]:
elo_df = load_elo()
print(f'Rows: {len(elo_df):,} | Columns: {elo_df.columns.tolist()}')
elo_df.head(3)

In [ ]:
# Current ELO snapshot (WC 2026 teams, as of tournament start)
elo_wc = elo_at_wc_start(elo_df)
current = elo_wc[elo_wc['year'] == 2026]
current_2026 = current[current['team'].isin(WC_2026_TEAMS)].sort_values('elo', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [ACCENT if e >= current_2026['elo'].quantile(0.75) else '#555'
          for e in current_2026['elo']]
ax.barh(current_2026['team'], current_2026['elo'], color=colors, height=0.7)
ax.set_title('ELO Ratings — WC 2026 Teams (June 2026)', color=ACCENT, fontsize=14, pad=12)
ax.set_xlabel('ELO Rating')
ax.axvline(current_2026['elo'].mean(), color=RED, linewidth=1.2,
           linestyle='--', label=f'Mean: {current_2026["elo"].mean():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('data/viz_elo_2026.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ELO over time — top 6 teams
top_teams = current_2026.nlargest(6, 'elo')['team'].tolist()
elo_top = elo_df[elo_df['team'].isin(top_teams)]
elo_top = elo_top[elo_top['date'] >= '2000-01-01']

colors_teams = [ACCENT, RED, '#4dabf7', '#f59f00', '#a9e34b', '#da77f2']

fig, ax = plt.subplots(figsize=(14, 6))
for i, team in enumerate(top_teams):
    subset = elo_top[elo_top['team'] == team].sort_values('date')
    ax.plot(subset['date'], subset['elo'],
            label=team, color=colors_teams[i], linewidth=1.8)

# Mark WC years
for year in [2002, 2006, 2010, 2014, 2018, 2022, 2026]:
    ax.axvline(pd.Timestamp(f'{year}-06-01'), color='#444', linewidth=0.8, linestyle=':')
    ax.text(pd.Timestamp(f'{year}-06-01'), ax.get_ylim()[0] + 30,
            str(year), fontsize=7, color='#666', ha='center')

ax.set_title('ELO History (2000–2026) — Top 6 Teams', color=ACCENT, fontsize=13, pad=10)
ax.set_ylabel('ELO Rating')
ax.legend(loc='upper left', framealpha=0.3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/viz_elo_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Dataset 4 — Correlation Analysis
Which features actually predict reaching the WC Final?

In [ ]:
# Build feature matrix — two outcomes: ko_score (0–6) + group_points
try:
    mgr       = load_manager_appearances()
    coach_exp = coach_wc_experience(mgr)
except Exception:
    coach_exp = None
    print('Coach experience unavailable — skipping that feature')

elo_wc         = elo_at_wc_start(elo_df)
feature_matrix = build_feature_matrix(ko_df, group_pts, elo_wc, coach_exp)

print(f'Shape: {feature_matrix.shape}')
print(f'Outcome cols: ko_score, group_points, champion, reached_final, reached_semi, reached_qf')
print(f'Feature cols: {[c for c in feature_matrix.select_dtypes(include=np.number).columns if c not in {"ko_score","group_points","champion","reached_final","reached_semi","reached_qf","year"}]}')
feature_matrix.head(4)

In [ ]:
# Correlations against all outcome dimensions
outcomes = {
    'champion':      'Winning the tournament',
    'reached_final': 'Reaching the Final',
    'reached_semi':  'Reaching the Semi-finals',
    'ko_score':      'KO score (0–6, continuous)',
    'group_points':  'Group stage points',
}

corr_results = {}
for col, label in outcomes.items():
    corr_results[col] = correlation_to_winning(feature_matrix, outcome_col=col)
    print(f'\n=== → {label} ===')
    print(corr_results[col].head(5).to_string(index=False))

In [ ]:
# Side-by-side correlation bar chart: ko_score vs group_points
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, outcome in zip(axes, ['ko_score', 'group_points']):
    sig = corr_results[outcome][corr_results[outcome]['significant']].copy()
    if sig.empty:
        ax.set_title(f'No significant correlations → {outcome}', color=RED)
        continue
    bar_colors = [ACCENT if r > 0 else RED for r in sig['pearson_r']]
    ax.barh(sig['feature'], sig['pearson_r'], color=bar_colors, height=0.6)
    ax.axvline(0, color='#555', linewidth=0.8)
    ax.set_title(f'Feature correlation → {outcome}\n(p<0.05 only)',
                 color=ACCENT, fontsize=11, pad=8)
    ax.set_xlabel('Pearson r')

plt.suptitle('What predicts WC success?', color=ACCENT, fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('data/viz_correlation_dual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: ELO vs ko_score + group_points (two panels)
plot_df = feature_matrix.dropna(subset=['elo'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: ELO vs KO score
sc1 = ax1.scatter(plot_df['elo'], plot_df['ko_score'],
                  alpha=0.4, c=plot_df['ko_score'], cmap='YlOrRd', s=35, vmin=0, vmax=6)
m, b = np.polyfit(plot_df['elo'].dropna(), plot_df.loc[plot_df['elo'].notna(), 'ko_score'], 1)
x_r = np.linspace(plot_df['elo'].min(), plot_df['elo'].max(), 100)
ax1.plot(x_r, m * x_r + b, color=ACCENT, linewidth=2)
ax1.set_yticks(range(7))
ax1.set_yticklabels(['DNQ', 'R32', 'R16', 'QF', 'SF', 'Final', 'Won'])
ax1.set_xlabel('Pre-tournament ELO')
ax1.set_title('ELO vs KO Score', color=ACCENT, fontsize=12, pad=8)
ax1.grid(True, alpha=0.3)
plt.colorbar(sc1, ax=ax1)

# Panel 2: ELO vs group_points
gp_plot = feature_matrix.dropna(subset=['elo', 'group_points'])
sc2 = ax2.scatter(gp_plot['elo'], gp_plot['group_points'],
                  alpha=0.4, c=gp_plot['group_points'], cmap='Blues', s=35)
m2, b2 = np.polyfit(gp_plot['elo'], gp_plot['group_points'], 1)
ax2.plot(x_r, m2 * x_r + b2, color=ACCENT, linewidth=2)
ax2.set_xlabel('Pre-tournament ELO')
ax2.set_ylabel('Group stage points')
ax2.set_title('ELO vs Group Stage Points', color=ACCENT, fontsize=12, pad=8)
ax2.grid(True, alpha=0.3)
plt.colorbar(sc2, ax=ax2)

plt.suptitle('ELO as predictor (1930–2018)', color=ACCENT, fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('data/viz_elo_vs_scores.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Dataset 5 — Fact Generator Output
Run pipeline on 2026 matchups.

In [ ]:
from pipeline import generate_matchup_facts, drought_since_last_title

# Drought chart — all WC 2026 teams
drought_df = drought_since_last_title(matches, WC_2026_TEAMS)
has_titles = drought_df[drought_df['type'] == 'drought'].sort_values('value', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [RED if v >= 20 else ACCENT for v in has_titles['value']]
ax.barh(has_titles['team'], has_titles['value'], color=bar_colors, height=0.7)
ax.set_title('Years Since Last WC Title — 2026 Teams', color=ACCENT, fontsize=13, pad=10)
ax.set_xlabel('Years without a title')
ax.axvline(24, color=RED, linewidth=1.2, linestyle='--', label='Brazil threshold (2002)')
ax.legend()
plt.tight_layout()
plt.savefig('data/viz_drought.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from pipeline import (load_international, load_matches, load_goalscorers,
                        load_geolocation, knockout_stage_score, generate_matchup_facts,
                        player_vs_team, player_career_goals)
intl    = load_international()
matches = load_matches()
goals   = load_goalscorers()
geo     = load_geolocation()
ko_df   = knockout_stage_score(matches)

teamA, teamB = "Brazil", "Morocco"
facts = generate_matchup_facts(intl, matches, teamA, teamB,
                                ko_df=ko_df, goals_df=goals, geo_df=geo)


In [ ]:
for f in facts:
      print(f"[{f['strength']:.2f}] {f['narrative']}")


In [ ]:
h2h = [f for f in facts if f["type"] == "all_time_h2h"][0]
print(h2h["by_type"])       # {"Friendly": 3, "Qualifier": 1}
print(h2h["last_3"])        # last 3 meetings with scores


In [ ]:
# Only form facts
form = [f for f in facts if f["type"] == "form_trajectory"]
for f in form:
    print(f["team"], f["trend"], f["delta"])  # "Brazil" "rising" 0.13


In [ ]:
# Continental records
cont = [f for f in facts if f["type"] == "continental_record"]
for f in cont:
    print(f["team"], f["cup"], f"{f['win_rate']:.0%}")  # "Morocco" "AFCON" "62%"


In [ ]:
# Altitude & distance
print([f for f in facts if f["type"] == "altitude_context"][0]["narrative"])
print([f for f in facts if f["type"] == "home_distance"][0]["narrative"])


In [ ]:
# Career goals for one player
print(player_career_goals(goals, intl, "Neymar"))


In [ ]:
# Goals specifically vs Morocco
print(player_vs_team(goals, intl, "Ronaldo", "Morocco"))


In [ ]:
# Who scored in Brazil vs Morocco history
from pipeline import top_scorers_in_matchup
print(top_scorers_in_matchup(goals, intl, "Brazil", "Morocco"))


In [ ]:
# Morocco's record in 1998 AFCON
rec = team_competition_record(intl, "Morocco", "African Cup", 1998)
print(rec["narrative"])  # "Morocco in African Cup 1998: 2W 1D 1L, 4–3 goals"

# Brazil at 2002 WC
rec = team_competition_record(intl, "Brazil", "World Cup", 2002)

# Any year (career total in competition)
rec = team_competition_record(intl, "Morocco", "African Cup")


In [ ]:
# Player goals in specific competition/year
print(player_competition_stats(goals, intl, "Ronaldo", "World Cup", 2002))
print(player_competition_stats(goals, intl, "Hakan Balta", "UEFA Euro"))


In [ ]:
# Game in Mexico City → altitude + distance to that specific venue
facts = generate_matchup_facts(intl, matches, "Brazil", "Morocco",
                                ko_df=ko_df, goals_df=goals, geo_df=geo,
                                city="Mexico City")

# Game in Dallas (low altitude, close to Mexico)
facts = generate_matchup_facts(..., city="Dallas")

# Available cities:
print(list(WC_2026_VENUE_COORDS.keys()))


In [ ]:
from sb_loader import (load_player_club_form, player_club_form_summary,
                        club_coappearances)

club_df = load_player_club_form()  # ~5-10min first run, cached after

# Player form last 4 seasons at club
print(player_club_form_summary(club_df, "Vinicius Junior"))
print(player_club_form_summary(club_df, "Hakim Ziyech"))

# Do two players share a club? How many seasons?
print(club_coappearances(club_df, "Vinicius Junior", "Jude Bellingham"))
# → "Vinicius Junior + Jude Bellingham: 2 seasons together at Real Madrid — established pair (UCL experience)"
